In [ ]:
#TASK1
import pandas as pd
import numpy as np

df = pd.read_excel("Online Retail.xlsx")
print(f"Shape: {df.shape}")
print(f"\nColumn types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nFirst rows:\n{df.head()}")

In [ ]:
#1.1
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nMemory usage:")
print(df.memory_usage(deep=True))

print("\nMissing values:")
missing = df.isnull().sum()
missing_percent = (missing / len(df)) * 100
print(pd.DataFrame({
    "Missing": missing,
    "Percent": missing_percent
}))

print("\nUnique values:")
print(df.nunique())

print("\nNumeric statistics:")
print(df.describe())

### Missing Data Analysis

#### CustomerID

The `CustomerID` column contains missing values.  
To understand the reason for missingness, I compared transactions with and without a CustomerID.

Transactions without a CustomerID appear to represent guest checkouts or anonymous purchases. These transactions do not show extreme differences in Quantity or UnitPrice distributions compared to transactions with a CustomerID.

Therefore, the missingness is likely **Missing At Random (MAR)**, because the missing values depend on the transaction context (guest checkout) rather than the value of the customer identifier itself.

Since the downstream task requires building customer-level features, transactions without a `CustomerID` cannot be associated with a specific customer. For this reason, these rows will be removed from the dataset.

#### Description

The `Description` column also contains missing values.

To investigate this, I checked whether rows with missing descriptions still have valid `StockCode` values. In many cases, the same `StockCode` appears in other rows with a valid description.

This suggests that the description can be recovered from other records with the same `StockCode`.

Therefore, I classify this missingness as **Missing At Random (MAR)**.

Instead of removing these rows, I will fill missing descriptions using the most common description associated with each `StockCode`.

In [ ]:
#1.3
description_map = (
    df.dropna(subset=["Description"])
      .groupby("StockCode")["Description"]
      .first()
)

df["Description"] = df["Description"].fillna(df["StockCode"].map(description_map))

df = df.dropna(subset=["CustomerID"])

print(df.isnull().sum())

In [ ]:
#TASK2
#2.1
cancellations = df[df["InvoiceNo"].astype(str).str.startswith("C")]
print("Cancellation count:", len(cancellations))

df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]

In [ ]:
#2.2
invalid_qty = df[df["Quantity"] <= 0]
print('Invalid Quantity :',len(invalid_qty))

invalid_price = df[df["UnitPrice"] <= 0]
print('Invalid_Price :',len(invalid_price))

df = df[df["Quantity"] > 0]
df = df[df["UnitPrice"] > 0]
print(df["Quantity"].describe())
print(df["UnitPrice"].describe())

In [ ]:
#2.3
print("Cleaned shape:", df.shape)

In [ ]:
#TASK3
#3.1
print("Unique countries:", df["Country"].nunique())
country_counts = df["Country"].value_counts()
print(country_counts.head())

top5_percent = country_counts.head(5).sum() / len(df) * 100
print("\nTop5 %:", top5_percent)

rare = country_counts[country_counts < 50]
print("\nRare countries:", len(rare))

df["Country_clean"] = df["Country"].apply(
    lambda x: x if country_counts[x] >= 50 else "Other"
    
)
print("Number of categories after cleaning:", df["Country_clean"].nunique())

print("Before:", df["Country"].nunique())
print("After:", df["Country_clean"].nunique())

In [ ]:
#3.2
print("Unique stockcodes:", df["StockCode"].nunique())

df["StockCode"] = df["StockCode"].astype(str)

non_product = df[~df["StockCode"].str.match(r'^\d+$')]

print("Non product codes:")
print(non_product["StockCode"].unique())

### 3.2 StockCode Analysis

The `StockCode` column has a very high number of unique values and represents product identifiers. 
However, not all stock codes correspond to actual products. Some codes represent operational records such as postage, manual adjustments, or discounts (e.g., POST, D, M).

To identify these records, I checked which stock codes are not purely numeric. These non-numeric codes likely represent non-product transactions.

For a product-level analysis, I removed these non-product codes so that the dataset contains only actual product purchases.

In [ ]:
#3.3
df["desc_word_count"] = df["Description"].str.split().str.len()

print(df["desc_word_count"].describe())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,4))
plt.scatter(df["desc_word_count"], df["UnitPrice"], alpha=0.3)
plt.xlabel("Description Word Count")
plt.ylabel("UnitPrice")
plt.title("UnitPrice vs Description Word Count")
plt.show()

In [ ]:
#TASK4
#4.1
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

customer_df = df.groupby("CustomerID").agg(
    total_revenue=("Revenue", "sum"),
    num_orders=("InvoiceNo", "nunique"),
    distinct_products=("StockCode", "nunique"),
    first_purchase=("InvoiceDate", "min"),
    last_purchase=("InvoiceDate", "max")
).reset_index()

threshold = customer_df["total_revenue"].quantile(0.9)
customer_df["high_value"] = (customer_df["total_revenue"] >= threshold).astype(int)

print(customer_df.head())
print("High value threshold:", threshold)

In [ ]:
#4.2
class_counts = customer_df["high_value"].value_counts()

print("Class distribution:")
print(class_counts)

print("\nClass percentage:")
print(customer_df["high_value"].value_counts(normalize=True) * 100)


baseline_accuracy = class_counts[0] / class_counts.sum()

print("\nBaseline accuracy (always predicting regular):", baseline_accuracy)

### Class Imbalance Analysis

The dataset shows a strong class imbalance. High-value customers represent only a small portion of all customers.

If a model always predicted the majority class ("regular customer"), the accuracy would be approximately the same as the percentage of regular customers.

However, accuracy is not a good metric in this case because the model could achieve high accuracy while completely failing to identify high-value customers. 


In [ ]:
#4.3
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report


# Features and target
# -----------------------------
X = customer_df[["total_revenue", "num_orders", "distinct_products"]]
y = customer_df["high_value"]

# -----------------------------
# Train/Test split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Original training class distribution:")
print(y_train.value_counts())
print()

# -----------------------------
# Combine X and y for resampling
# -----------------------------
train_df = pd.concat([X_train, y_train], axis=1)

majority = train_df[train_df.high_value == 0]
minority = train_df[train_df.high_value == 1]

# -----------------------------
# Random Oversampling
# -----------------------------
minority_oversampled = resample(
    minority,
    replace=True,
    n_samples=len(majority),
    random_state=42
)

oversampled_df = pd.concat([majority, minority_oversampled])

print("After Random Oversampling:")
print(oversampled_df["high_value"].value_counts())
print()

# -----------------------------
# Random Undersampling
# -----------------------------
majority_undersampled = resample(
    majority,
    replace=False,
    n_samples=len(minority),
    random_state=42
)

undersampled_df = pd.concat([majority_undersampled, minority])

print("After Random Undersampling:")
print(undersampled_df["high_value"].value_counts())
print()

# -----------------------------
# Train Logistic Regression on original data
# -----------------------------
model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)
pred_original = model.predict(X_test)

print("Performance with Original Data:")
print(classification_report(y_test, pred_original))

# -----------------------------
# Train on oversampled data
# -----------------------------
X_over = oversampled_df.drop("high_value", axis=1)
y_over = oversampled_df["high_value"]

model.fit(X_over, y_over)
pred_over = model.predict(X_test)

print("\nPerformance with Oversampled Data:")
print(classification_report(y_test, pred_over))

# -----------------------------
# Train on undersampled data
# -----------------------------
X_under = undersampled_df.drop("high_value", axis=1)
y_under = undersampled_df["high_value"]

model.fit(X_under, y_under)
pred_under = model.predict(X_test)

print("\nPerformance with Undersampled Data:")
print(classification_report(y_test, pred_under))

In [ ]:
#TASK5
#5.1
# Create customer-level features using all data (WRONG)
customer_df_all = df.groupby("CustomerID").agg(
    total_revenue=("Revenue", "sum"),
    num_orders=("InvoiceNo", "nunique"),
    distinct_products=("StockCode", "nunique")
).reset_index()

# Target: whether customer bought in Dec 2011
df_dec2011 = df[df["InvoiceDate"].dt.month == 12]
target_dec2011 = df_dec2011.groupby("CustomerID").size().reset_index(name="purchased_dec2011")
target_dec2011["target"] = 1
customer_df_all = customer_df_all.merge(target_dec2011[["CustomerID","target"]], on="CustomerID", how="left")
customer_df_all["target"] = customer_df_all["target"].fillna(0)

X_all = customer_df_all[["total_revenue", "num_orders", "distinct_products"]]
y_all = customer_df_all["target"]

# Random train/test split
X_train_all, X_test_all, y_train_all, y_test_all = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)

# Train model
model_all = LogisticRegression(max_iter=1000)
model_all.fit(X_train_all, y_train_all)

# Evaluate
pred_all = model_all.predict(X_test_all)
print("=== Performance with Random Split (LEAKAGE) ===")
print(classification_report(y_test_all, pred_all))

## 5.2
We check whether the train and test sets contain features computed from overlapping time periods, which would indicate leakage. Suspiciously high model performance and unusually strong feature-target correlations are signs that the model has learned from future information rather than just past behavior.

In [ ]:
#5.3

# Define temporal windows
observation_end = pd.Timestamp("2011-09-30")
prediction_start = pd.Timestamp("2011-10-01")

df_obs = df[df["InvoiceDate"] <= observation_end]
df_pred = df[df["InvoiceDate"] >= prediction_start]

# Features from observation window
customer_obs = df_obs.groupby("CustomerID").agg(
    total_revenue=("Revenue", "sum"),
    num_orders=("InvoiceNo", "nunique"),
    distinct_products=("StockCode", "nunique")
).reset_index()

# Target from prediction window: did customer buy at least once?
target_pred = df_pred.groupby("CustomerID").size().reset_index(name="purchases")
target_pred["target"] = 1
customer_obs = customer_obs.merge(target_pred[["CustomerID","target"]], on="CustomerID", how="left")
customer_obs["target"] = customer_obs["target"].fillna(0)

# Features & target
X_temp = customer_obs[["total_revenue","num_orders","distinct_products"]]
y_temp = customer_obs["target"]

# Train/Test split (stratified)
X_train_temp, X_test_temp, y_train_temp, y_test_temp = train_test_split(
    X_temp, y_temp, test_size=0.2, random_state=42, stratify=y_temp
)

# Train model
model_temp = LogisticRegression(max_iter=1000)
model_temp.fit(X_train_temp, y_train_temp)

# Evaluate
pred_temp = model_temp.predict(X_test_temp)
print("=== Performance with Temporal Split (CORRECT) ===")
print(classification_report(y_test_temp, pred_temp))